In [1]:
import pandas as pd
import numpy as np
import json


### Task 1: Build a Relational Schema

Model an online movie rental service. Design a normalized relational schema using pandas DataFrames for the following entities:

**Step 1:** Create these tables:
- `customers` with columns: `customer_id`, `name`, `email`, `city`, `signup_date`
- `movies` with columns: `movie_id`, `title`, `genre`, `release_year`, `rating` (1-5 scale)
- `rentals` with columns: `rental_id`, `customer_id`, `movie_id`, `rental_date`, `return_date`, `price`

Populate with at least 8 customers, 10 movies, and 25 rentals. Make sure some customers have multiple rentals and some movies are rented multiple times.





In [2]:
customers = pd.DataFrame({
    "customer_id": [1,2,3,4,5,6,7,8],
    "name": [
        "Ali Karimov", "Narmin Aliyeva", "Elvin Mammadov", "Leyla Hasanova",
        "Rashad Aliyev", "Aysel Quliyeva", "Murad Abbasov", "Kamal Rzayev"
    ],
    "email": [
        "ali@gmail.com", "narmin@gmail.com", "elvin@gmail.com", "leyla@gmail.com",
        "rashad@gmail.com", "aysel@gmail.com", "murad@gmail.com", "kamal@gmail.com"
    ],
    "city": [
        "Baku", "Ganja", "Baku", "Sumqayit",
        "Baku", "Ganja", "Shaki", "Baku"
    ],
    "signup_date": pd.to_datetime([
        "2023-01-15", "2023-02-10", "2023-03-05", "2023-04-20",
        "2023-05-01", "2023-06-18", "2023-07-12", "2023-08-30"
    ])
})

In [3]:
movies = pd.DataFrame({
    "movie_id": range(1, 11),
    "title": [
        "Inception", "Interstellar", "Titanic", "Avengers",
        "Joker", "Matrix", "Gladiator", "Avatar",
        "Forrest Gump", "Dark Knight"
    ],
    "genre": [
        "Sci-Fi", "Sci-Fi", "Romance", "Action",
        "Drama", "Sci-Fi", "Action", "Fantasy",
        "Drama", "Action"
    ],
    "release_year": [
        2010, 2014, 1997, 2012, 2019,
        1999, 2000, 2009, 1994, 2008
    ],
    "rating": [5, 5, 4, 5, 4, 5, 4, 4, 5, 5]
})

In [4]:
rentals = pd.DataFrame({
    "rental_id": range(1, 26),

    "customer_id": [
        1,2,3,4,1,5,6,2,3,7,
        8,4,1,6,5,2,7,3,8,4,
        1,5,6,2,3
    ],

    "movie_id": [
        1,2,3,4,5,6,7,1,2,3,
        4,5,6,7,8,9,10,1,2,3,
        4,5,6,7,8
    ],

    "rental_date": pd.to_datetime([
        "2024-01-01","2024-01-03","2024-01-05","2024-01-07","2024-01-10",
        "2024-01-12","2024-01-15","2024-01-17","2024-01-20","2024-01-22",
        "2024-01-25","2024-01-27","2024-02-01","2024-02-03","2024-02-05",
        "2024-02-08","2024-02-10","2024-02-12","2024-02-15","2024-02-18",
        "2024-02-20","2024-02-22","2024-02-25","2024-02-27","2024-03-01"
    ]),

    "return_date": pd.to_datetime([
        "2024-01-05","2024-01-07","2024-01-09","2024-01-11","2024-01-14",
        "2024-01-16","2024-01-19","2024-01-21","2024-01-24","2024-01-26",
        "2024-01-29","2024-01-31","2024-02-05","2024-02-07","2024-02-09",
        "2024-02-12","2024-02-14","2024-02-16","2024-02-19","2024-02-22",
        "2024-02-24","2024-02-26","2024-02-28","2024-03-02","2024-03-05"
    ]),

    "price": [
        4.99, 5.99, 3.99, 6.99, 4.49,
        5.49, 6.49, 4.99, 5.99, 3.99,
        6.99, 4.49, 5.49, 6.49, 4.99,
        3.99, 5.99, 4.99, 5.99, 3.99,
        6.99, 4.49, 5.49, 6.49, 4.99
    ]
})

**Step 2:** Write and execute these queries using pandas operations:
1. Find all movies rented by a specific customer (requires joining rentals with movies).
2. Find the top 5 most-rented movies (group and count).
3. Compute the total revenue per genre (join rentals with movies, group by genre, sum price).
4. Find customers who have never rented a movie rated below 3 (multi-step filtering).

In [5]:
rentals_movies=pd.merge(rentals,movies,on="movie_id",how="inner")
result=rentals_movies[rentals_movies["customer_id"]==3]["title"]
print(result)

2          Titanic
8     Interstellar
17       Inception
24          Avatar
Name: title, dtype: object


In [6]:
top5=rentals_movies.groupby(["movie_id","title"]).size().reset_index(name="rental_count").sort_values("rental_count",ascending=False).head(5)
top5

,movie_id,title,rental_count
0,1,Inception,3
1,2,Interstellar,3
2,3,Titanic,3
3,4,Avengers,3
4,5,Joker,3


In [7]:
revenue_per_genre=rentals_movies.groupby("genre")["price"].sum().reset_index(name="total_revenue")
revenue_per_genre

,genre,total_revenue
0,Action,46.43
1,Drama,17.46
2,Fantasy,9.98
3,Romance,11.97
4,Sci-Fi,49.41


In [8]:
low_rating_customers=rentals_movies[rentals_movies["rating"]<3]["customer_id"].unique()

In [9]:
result=customers[ ~customers["customer_id"].isin(low_rating_customers)]


**Step 3:** Demonstrate normalization. Show what the data would look like as a single denormalized table (join all three tables). Write a markdown cell explaining what redundancy appears and why the normalized version is preferable for updates.


In [10]:
cities = pd.DataFrame({
    "city_id": range(1,customers["city"].nunique()+1),
    "city_name": customers["city"].unique()
})

genres=pd.DataFrame({
    "genre_id": range(1,movies["genre"].nunique()+1),
    "genre_name": movies["genre"].unique()
})


In [11]:
movies_normalized=movies.merge(genres,left_on="genre",right_on="genre_name")
movies_normalized=movies_normalized.drop(columns=["genre","genre_name"])

In [12]:
customers_normalized=customers.merge(cities,left_on="city",right_on="city_name")
customers_normalized=customers_normalized.drop(columns=["city","city_name"])                                           

In [13]:
denormalized_df = rentals.merge(customers, on='customer_id').merge(movies, on='movie_id')

######
In the denormalized table, we see significant data redundancy because customer details (like names and emails) and movie details (like titles and genres) are repeated across every rental row. For example, if a customer rents five movies, their personal info is stored five times, wasting memory. The normalized version is much better for updates because it ensures data integrity; if a customer changes their email or a movie price is updated, you only need to change it in one single place (the parent table) rather than hunting through thousands of rental records. 

### Task 2: Model the Same Data as Documents

Represent the same movie rental data using the document model.

**Step 1:** Create a list of customer documents (Python dictionaries) where each document contains the customer's information and a nested list of their rental history, with movie details embedded in each rental entry.


In [14]:
grouped=denormalized_df.groupby("customer_id")
customer_documents=[]
for customer_id,group in grouped:
    row0=group.iloc[0]
    rental_list=[]
    for _,row in group.iterrows():
        rental={
            "rental_id":row["rental_id"],
            "movie_id":row["movie_id"],
            "genre":row["genre"],
            "title":row["title"],
            "rental_date": str(row["rental_date"]),
            "return_date": str(row["return_date"]),
            "price": row["price"],
            "rating":row["rating"]
        }
        rental_list.append(rental)
    customer_doc={
        "customer_id": customer_id,
        "name": row0["name"],
        "email": row0["email"],
        "city": row0["city"],
        "signup_date": str(row0["signup_date"]),
        "rentals": rental_list
    }
    customer_documents.append(customer_doc)
print(json.dumps(customer_documents, indent=2))            

[
  {
    "customer_id": 1,
    "name": "Ali Karimov",
    "email": "ali@gmail.com",
    "city": "Baku",
    "signup_date": "2023-01-15 00:00:00",
    "rentals": [
      {
        "rental_id": 1,
        "movie_id": 1,
        "genre": "Sci-Fi",
        "title": "Inception",
        "rental_date": "2024-01-01 00:00:00",
        "return_date": "2024-01-05 00:00:00",
        "price": 4.99,
        "rating": 5
      },
      {
        "rental_id": 5,
        "movie_id": 5,
        "genre": "Drama",
        "title": "Joker",
        "rental_date": "2024-01-10 00:00:00",
        "return_date": "2024-01-14 00:00:00",
        "price": 4.49,
        "rating": 4
      },
      {
        "rental_id": 13,
        "movie_id": 6,
        "genre": "Sci-Fi",
        "title": "Matrix",
        "rental_date": "2024-02-01 00:00:00",
        "return_date": "2024-02-05 00:00:00",
        "price": 5.49,
        "rating": 5
      },
      {
        "rental_id": 21,
        "movie_id": 4,
        "genre": "A

**Step 2:** Execute the same four queries from Task 1 on the document collection:
1. Find all movies rented by a specific customer.
2. Find the top 5 most-rented movies.
3. Compute total revenue per genre.
4. Find customers who have never rented a movie rated below 3.

In [15]:
#1.
movies=[]
for doc in customer_documents:
    if doc["customer_id"]==3:
        for rental in doc["rentals"]:
            movies.append(rental["title"])
movies

['Titanic', 'Interstellar', 'Inception', 'Avatar']

In [16]:
#2.
from collections import Counter
top5=Counter(rental["title"] for doc in customer_documents for rental in doc["rentals"]).most_common(5)
top5

[('Inception', 3),
 ('Joker', 3),
 ('Matrix', 3),
 ('Avengers', 3),
 ('Interstellar', 3)]

In [17]:
#3
revenue_per_genre={}
for customer in customer_documents:
    for rental in customer["rentals"]:
        genre=rental["genre"]
        price=rental["price"]

        if genre in revenue_per_genre:
            revenue_per_genre[genre]+=price
        else:
            revenue_per_genre[genre]=price

revenue_per_genre

{'Sci-Fi': 49.41000000000001,
 'Drama': 17.46,
 'Action': 46.43000000000001,
 'Romance': 11.97,
 'Fantasy': 9.98}

In [18]:
#4
high_rating_customers=[]
for customer in customer_documents:
    low_rating_customers=False
    for rental in customer["rentals"]:
        if rental["rating"]<3:
            low_rating_customers=True
            break
    if not low_rating_customers:
        high_rating_customers.append(customer["name"])
high_rating_customers

['Ali Karimov',
 'Narmin Aliyeva',
 'Elvin Mammadov',
 'Leyla Hasanova',
 'Rashad Aliyev',
 'Aysel Quliyeva',
 'Murad Abbasov',
 'Kamal Rzayev']

**Step 3:** Write a markdown cell comparing the document queries to the relational queries. Which queries were easier to write? Which were harder? For which query pattern does the document model have a clear advantage?


In my opinion, the relational model is more convenient because it is vectorized and allows calculations to be done easily and quickly using functions like `groupby`, `sum`, and `count`. Many queries can be written in one line, which makes the code shorter and more efficient.

The document model usually requires using loops, so the code becomes longer and takes more time to write. However, because everything is written step by step with loops, it is easier to see and understand how the data is processed. For this reason, the document model is simpler in terms of logic and learning.

Overall, the relational model is better for analysis and calculations, while the document model is better for clearly understanding the process and working with customer-based data.

### Task 3: Model Relationships as a Graph

Build a graph representation of the rental data to answer relationship-based questions.

**Step 1:** Create a graph using Python dictionaries where:
- Nodes represent customers, movies, and genres.
- Edges represent relationships: `rented` (customer → movie), `belongs_to` (movie → genre), `lives_in` (customer → city).

```python
# Suggested structure
nodes = {
    "c001": {"type": "customer", "name": "Alice"},
    "m001": {"type": "movie", "title": "Inception"},
    "action": {"type": "genre", "name": "Action"},
    # ... more nodes
}

edges = [
    ("c001", "rented", "m001"),
    ("m001", "belongs_to", "action"),
    # ... more edges
]
```




In [19]:
nodes={
    "c001": {"type": "customer", "name": "Ali"},
    "c002": {"type": "customer", "name": "Nigar"},
    "c003": {"type": "customer", "name": "Murad"},

    "m001": {"type": "movie", "title": "Inception"},
    "m002": {"type": "movie", "title": "Titanic"},
    "m003": {"type": "movie", "title": "Avatar"},

    "g001": {"type": "genre", "name": "Sci-Fi"},
    "g002": {"type": "genre", "name": "Drama"},

    "city_baku": {"type": "city", "name": "Baku"},
    "city_ganja": {"type": "city", "name": "Ganja"}
}
edges=[
    ("c001", "rented", "m001"),
    ("c001", "rented", "m002"),
    ("c002", "rented", "m001"),
    ("c003", "rented", "m003"),

    ("m001", "belongs_to", "g001"),
    ("m002", "belongs_to", "g002"),
    ("m003", "belongs_to", "g001"),

    ("c001", "lives_in", "city_baku"),
    ("c002", "lives_in", "city_ganja"),
    ("c003", "lives_in", "city_baku")
]

**Step 2:** Write traversal functions to answer:
1. "Which movies did customer X rent?" — one hop from customer through `rented` edges.
2. "Which genres does customer X prefer?" — two hops: customer → movies → genres, then count.
3. "Which customers share a genre preference with customer X?" — three hops: customer X → movies → genres → other customers who rented movies in the same genre.
4. "Recommend movies for customer X" — find movies in their preferred genres that they have not yet rented.


In [20]:
#1
def get_movies_rented_by(customer_id,nodes,edges):
    movies=[]
    for src,relation,dst in edges:
        if src==customer_id and relation== "rented":
            movies.append(nodes[dst]["title"])
    return movies
        

In [21]:
result = get_movies_rented_by("c001", nodes, edges)
print(result)

['Inception', 'Titanic']


In [22]:
#2
def get_preferred_genres(customer_id, nodes, edges):
    genre_count={}
    for src,relation,movie_id in edges:
        if src==customer_id and relation=="rented":
            for src2,relation2,genre_id in edges:
                if src2==movie_id and relation2=="belongs_to":
                    genre_name=nodes[genre_id]["name"]
                    if genre_name in genre_count:
                         genre_count[genre_name]+=1
                    else:
                        genre_count[genre_name]=1
    return genre_count                                                     

In [23]:
result = get_preferred_genres("c001", nodes, edges)
print(result)

{'Sci-Fi': 1, 'Drama': 1}


In [24]:
#3
def get_customers_with_same_genre(customer_id, nodes, edges):
    shared_customers = set()
    for src,rel,movie_id in edges:
        if src==customer_id and rel=="rented":
            for src2,rel2,genre_id in edges:
                if src2==movie_id and rel2=="belongs_to":
                    for src3,rel3,movie_id2 in edges:
                        if movie_id2==genre_id and rel3=="belongs_to":
                            for src4,rel4,customer_id2 in edges:
                                if src4!=customer_id and customer_id2==src3 and rel4=="rented":
                                    shared_customers.add(src4)
                                    
    result = [nodes[c]["name"] for c in shared_customers]
    return result
                

In [25]:
result = get_customers_with_same_genre("c001", nodes, edges)
print(result)

['Murad', 'Nigar']


In [26]:
#4
def recommend_movies(customer_id, nodes, edges):
    rented_movies = set()
    preferred_genres = set() 
    recommendations = set() 

    for src,rel,dst in edges:
        if src==customer_id and rel=="rented":
            rented_movies.add(dst)
            
    for movie_id in rented_movies:
        for src,rel,dst in edges:
            if src==movie_id and rel=="belongs_to":
                preferred_genres.add(dst)
    
    for src,rel,dst in edges:
        if rel=="belongs_to" and dst in preferred_genres:
            movie_id=src
            if movie_id not in rented_movies:
                recommendations.add(nodes[movie_id]["title"])
                
    return list(recommendations)
                

In [27]:
result = recommend_movies("c001", nodes, edges)
print(result)

['Avatar']



**Step 3:** Write a markdown cell explaining why the recommendation query (Task 3, question 4) would be difficult to express in SQL or with documents. What makes the graph model natural for this kind of traversal?


In my opinion, the recommendation query is difficult to write in SQL and also in the document model.

In the relational model, this query requires many JOIN operations between customers, rentals, movies, and genres. When the data grows, these joins become complex and harder to read. It is not very natural to think in terms of many tables when we want to follow relationships step by step.

In the document model, the data is nested, but finding recommendations still requires multiple loops. We need to go from customer to rentals, then to genres, then to other movies, and finally remove the already rented ones. This makes the code longer and more complicated.

In the graph model, this query is much more natural because the data is already connected with edges. We can simply follow paths like:
customer → movie → genre → movie.
This matches exactly how we think about recommendations.

### Task 4: OLTP vs OLAP Workload Analysis

This task asks you to reason about how different database types handle different workloads.

**Step 1:** Create a DataFrame representing a log of 5,000 e-commerce transactions with columns: `transaction_id`, `customer_id`, `product_id`, `amount`, `payment_method`, `timestamp`, `status` (completed/refunded/pending).






n = 5000
np.random.seed(42)
transaction_id = range(1, n + 1)
customer_id = np.random.randint(1000, 1100, n)
product_id = np.random.randint(200, 300, n)
amount = np.round(np.random.uniform(5, 500, n), 2)

payment_method = np.random.choice(["Credit Card", "Debit Card", "PayPal", "Bank Transfer", "Cash"],n)

timestamp = pd.date_range("2025-01-01", periods=n, freq="min")

status = np.random.choice(["completed", "refunded", "pending"],n,p=[0.7, 0.15, 0.15])

transactions_df = pd.DataFrame({
    "transaction_id": transaction_id,
    "customer_id": customer_id,
    "product_id": product_id,
    "amount": amount,
    "payment_method": payment_method,
    "timestamp": timestamp,
    "status": status
})

transactions_df

**Step 2:** Simulate OLTP-style operations:
1. Look up a single transaction by its ID (simulate a point query).
2. Insert a new transaction (append a row).
3. Update the status of a transaction from "pending" to "completed."
4. Check that the transaction is valid: the amount must be positive and the status must be one of the allowed values (simulate a consistency check).


In [29]:
#1
result = transactions_df[transactions_df["transaction_id"] == 2500]
result

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
2499,2500,1041,264,320.81,PayPal,2025-01-02 17:39:00,completed


In [30]:
#2
new_transaction = {
    "transaction_id": 5001,
    "customer_id": 1050,
    "product_id": 250,
    "amount": 129.99,
    "payment_method": "Credit Card",
    "timestamp": pd.Timestamp.now(),
    "status": "completed"
}
transactions_df=pd.concat([transactions_df,pd.DataFrame([new_transaction])],ignore_index=True)
transactions_df.tail(1)

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
5000,5001,1050,250,129.99,Credit Card,2026-03-04 22:56:36.169663,completed


In [33]:
#3
update_id = 4996
transactions_df.loc[transactions_df["transaction_id"] == update_id,"status"] = "completed"
transactions_df[transactions_df["transaction_id"] == update_id]

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
4995,4996,1048,215,441.19,Debit Card,2025-01-04 11:15:00,completed


In [38]:
#4
allowed_status = ["completed", "refunded", "pending"]
invalid_transactions=transactions_df[(transactions_df["amount"]<=0)|(~transactions_df["status"].isin(allowed_status))]


**Step 3:** Simulate OLAP-style operations on the same data:
1. Compute total revenue by payment method.
2. Find the average transaction amount by month.
3. Identify the top 10 customers by total spending.
4. Compute the refund rate (percentage of refunded transactions).

In [43]:
#1
completed_df = transactions_df[transactions_df["status"] == "completed"]
revenue_by_payment = completed_df.groupby("payment_method")["amount"].sum()
revenue_by_payment

payment_method
Bank Transfer    168275.68
Cash             182711.50
Credit Card      188468.55
Debit Card       176666.93
PayPal           176484.09
Name: amount, dtype: float64

In [42]:
#2
transactions_df["timestamp"]=pd.to_datetime(transactions_df["timestamp"])
transactions_df["month"]=transactions_df["timestamp"].dt.to_period("M")
avg_by_month=transactions_df.groupby("month")["amount"].mean()
avg_by_month

month
2025-01    249.948046
2026-03    129.990000
Freq: M, Name: amount, dtype: float64

In [45]:
#3
top_customers=completed_df.groupby("customer_id")["amount"].sum().sort_values(ascending=False).head(10)
top_customers

customer_id
1050    13011.46
1014    12352.49
1024    12224.36
1020    11857.33
1016    11696.07
1086    11668.99
1046    11614.04
1002    11463.60
1025    11415.70
1098    11337.90
Name: amount, dtype: float64

In [50]:
#4
total_count=len(transactions_df)
refunded_count=len(transactions_df[transactions_df["status"]=="refunded"])
refunded_perc=(refunded_count/total_count)*100
refunded_perc

14.437112577484504

**Step 4:** Write a markdown cell explaining: Why would OLTP operations typically use row-major storage while OLAP operations use column-major storage? How does the access pattern of each workload type map to the memory layout that serves it best?

OLTP and OLAP systems have different goals, so they use different data storage layouts.
In OLTP systems, most operations work on a single record at a time, such as inserting a new transaction, updating a status, or looking up one transaction by its ID. These operations usually need all the information from one row. Because of this, row-major storage is more efficient, since all the columns of a single row are stored next to each other in memory. This makes reading or updating one record fast.
In OLAP systems, the main goal is to analyze large amounts of data using aggregations such as SUM, COUNT, or AVG. These queries usually access only a few columns but across many rows, for example, calculating total revenue or average amounts. Column-major storage is better in this case because values from the same column are stored together. This allows faster scanning, better cache usage, and more effective compression.
So, OLTP workloads benefit from row-major storage because they frequently access complete rows, while OLAP workloads benefit from column-major storage because they frequently scan individual columns. Each storage layout matches the typical access pattern of its workload type, which is why they are used in different systems.


### Task 5: Choosing the Right Model

For each of the following real-world scenarios, recommend a data model (relational, document, or graph) and a processing paradigm (OLTP or OLAP). Write 3-4 sentences justifying each choice.

1. A hospital patient record system where doctors need to quickly look up a patient's complete medical history.
2. A social network that needs to suggest "friends of friends" connections.
3. A financial reporting system that computes monthly revenue breakdowns across product categories and regions.
4. An IoT platform that receives sensor readings from 10,000 devices and needs to detect anomalies in real time.
5. A content management system where each article has a different set of metadata fields.


###### 1.A hospital patient record system where doctors need to quickly look up a patient's complete medical history.
A relational data model is suitable for this system because patient records are highly structured and require strong consistency and integrity. Medical data is usually stored in well-defined tables such as patients, visits, diagnoses, and treatments. OLTP is appropriate because doctors frequently perform fast lookups and updates on individual patient records. The system focuses on real-time access rather than large-scale analysis.

###### 2.A social network that needs to suggest "friends of friends" connections.
A graph data model is the best choice because social networks are based on relationships between users. Connections such as friends, followers, and mutual contacts can be represented naturally as nodes and edges. This makes multi-hop queries like “friends of friends” easy and efficient. OLTP fits this scenario since the system needs to handle many real-time queries and updates.

###### 3.A financial reporting system that computes monthly revenue breakdowns across product categories and regions.
A relational data model works well here because financial data is structured and stored in tables such as sales, products, and regions. These systems require strong accuracy and consistency. OLAP is the right processing paradigm because the main goal is to compute summaries and aggregates over large datasets. Queries like monthly revenue and category breakdowns are typical analytical workloads.

###### 4.An IoT platform that receives sensor readings from 10,000 devices and needs to detect anomalies in real time.
A document data model is suitable because sensor data can vary in structure and may include different fields for different devices. Storing each reading as a document provides flexibility. OLTP is appropriate since the system processes high-speed incoming data and needs to react quickly to anomalies. The focus is on real-time ingestion and monitoring rather than long-term analysis.

###### 5.A content management system where each article has a different set of metadata fields.
A document data model is ideal because each article may have a different set of metadata fields. Documents allow storing variable and nested data without a fixed schema. OLTP fits this use case because users frequently create, update, and retrieve individual articles. The system mainly handles operational queries rather than complex analytics.
